## Part 2: Lipinski Descriptors

### Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
sns.set(style='ticks')

df_clean = pd.read_csv("df_clean.csv")
print("Loaded:", df_clean.shape)

### Install and Import RDKit

In [ ]:
!pip install rdkit

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

### Compute Lipinski Descriptors

In [ ]:
def lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return pd.Series([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol)
    ])

lipinski_desc = df_clean["canonical_smiles"].apply(lipinski)
lipinski_desc.columns = ["MW", "LogP", "NumHDonors", "NumHAcceptors"]

df_combined = pd.concat([df_clean, lipinski_desc], axis=1)
df_combined.to_csv("df_lipinski.csv", index=False)
print(df_combined.head())

### Boxplots: Lipinski Descriptors vs Bioactivity Class

In [ ]:
for col in ["MW", "LogP", "NumHDonors", "NumHAcceptors"]:
    plt.figure(figsize=(5.5, 5.5))
    sns.boxplot(x="bioactivity_class", y=col, data=df_combined, hue="bioactivity_class")
    plt.xlabel("Bioactivity Class")
    plt.ylabel(col)
    plt.title(f"{col} vs Bioactivity Class")
    plt.savefig(f"boxplot_{col}.pdf")
    plt.show()

### Scatter Plot: MW vs LogP

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(
    x="MW", y="LogP", data=df_combined,
    hue="bioactivity_class", size="pIC50",
    alpha=0.7, edgecolor="black"
)
plt.xlabel("MW")
plt.ylabel("LogP")
plt.title("MW vs LogP by Bioactivity")
plt.savefig("scatter_MW_vs_LogP.pdf")
plt.show()

### Mann–Whitney U Test

In [ ]:
def mannwhitney(feature):
    active   = df_combined[df_combined.bioactivity_class == "active"][feature]
    inactive = df_combined[df_combined.bioactivity_class == "inactive"][feature]
    stat, p  = mannwhitneyu(active, inactive)
    print(f"{feature}")
    print("Statistic:", stat)
    print("p-value:", p)
    print("------------------")

features = ["pIC50", "MW", "LogP", "NumHDonors", "NumHAcceptors"]
for f in features:
    mannwhitney(f)